#### Credit Card Fraud Detection Model


In [1]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve


c:\Users\Dilsh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
#configaration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
sns.set_style(style='darkgrid')


##### lode data set


In [3]:
df=pd.read_csv(r"C:\Users\Dilsh\Desktop\Machin_Learning\Data_sets\creditcard.csv")

#### EDA


In [4]:
df.shape

(284807, 31)

In [5]:
df.head(2)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.000,-1.360,-0.073,2.536,1.378,-0.338,0.462,0.240,0.099,0.364,0.091,-0.552,-0.618,-0.991,-0.311,1.468,-0.470,0.208,0.026,0.404,0.251,-0.018,0.278,-0.110,0.067,0.129,-0.189,0.134,-0.021,149.620,0
1,0.000,1.192,0.266,0.166,0.448,0.060,-0.082,-0.079,0.085,-0.255,-0.167,1.613,1.065,0.489,-0.144,0.636,0.464,-0.115,-0.183,-0.146,-0.069,-0.226,-0.639,0.101,-0.340,0.167,0.126,-0.009,0.015,2.690,0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [7]:
# check missing values
df.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [8]:
# check the encoded missing values
for col in df.columns:
    print(df[col].value_counts().head(5))

Time
163152.000    36
64947.000     26
68780.000     25
3767.000      21
3770.000      20
Name: count, dtype: int64
V1
1.246    77
2.056    77
2.053    62
1.302    60
2.040    53
Name: count, dtype: int64
V2
0.167     77
-0.327    77
0.090     62
-0.607    60
-0.147    53
Name: count, dtype: int64
V3
0.488     77
-2.752    77
-1.682    62
-0.682    60
-2.956    53
Name: count, dtype: int64
V4
0.635     77
-0.842    77
0.454     62
-1.905    60
-0.578    53
Name: count, dtype: int64
V5
-0.563    77
2.463     77
0.298     62
1.327     60
2.609     53
Name: count, dtype: int64
V6
-1.011    77
3.174     77
-0.954    62
3.436     60
3.143     53
Name: count, dtype: int64
V7
0.015     77
-0.432    77
0.152     62
-1.145    60
-0.417    53
Name: count, dtype: int64
V8
-0.160    77
0.728     77
-0.207    62
0.959     60
0.784     53
Name: count, dtype: int64
V9
0.170    77
0.609    77
0.587    62
1.671    60
0.360    53
Name: count, dtype: int64
V10
-0.045    77
-0.075    77
-0.362    62
-1.02

In [9]:
# check duplicate values
df.duplicated().sum()

np.int64(1081)

In [10]:
duplicated_columns = []

for col in df.columns:
    if df[col].duplicated().any():
        duplicated_columns.append(col)

print("Columns with duplicates:", duplicated_columns)

Columns with duplicates: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [11]:
df.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='str')

In [19]:
print(df['Class'].value_counts())
print("--"*50)
print(df['Class'].value_counts(normalize=True)*100)


Class
0    283253
1       492
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
Class
0   99.827
1    0.173
Name: proportion, dtype: float64


##### this data set is highly imbalanced data set

In [14]:
# check hoe may duplicate values in eatch class
print("Number of duplicate values in Class 0:", df[df['Class'] == 0].duplicated().sum())
print("Number of duplicate values in Class 1:", df[df['Class'] == 1].duplicated().sum())

Number of duplicate values in Class 0: 1062
Number of duplicate values in Class 1: 19


In [15]:
# drop duplicate values in class 0
df = df[~((df['Class'] == 0) & (df.duplicated()))]

In [ ]:
# imbalanced ratio of dataset
n_fraud=(df['Class']==1).sum()
n_non_fraud=(df['Class']==0).sum()
# calculate the ratio of fraud to non-fraud transactions
print(round(n_non_fraud/max(n_fraud, 1), 4))


575.7175


##### That means for every 1 fraud case, there are about 576 normal transactions.

In [25]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Time,283745.000,94810.630,47479.710,0.000,54208.000,84695.000,139297.000,172792.000
V1,283745.000,0.005,1.951,-56.408,-0.916,0.020,1.316,2.455
V2,283745.000,-0.004,1.649,-72.716,-0.600,0.064,0.800,22.058
V3,283745.000,0.001,1.515,-48.326,-0.890,0.180,1.027,9.383
V4,283745.000,-0.003,1.415,-5.683,-0.850,-0.022,0.740,16.875
V5,283745.000,0.001,1.379,-113.743,-0.690,-0.054,0.612,34.802
V6,283745.000,-0.001,1.332,-26.161,-0.769,-0.275,0.397,73.302
V7,283745.000,0.001,1.236,-43.557,-0.553,0.041,0.570,120.589
V8,283745.000,-0.001,1.190,-73.217,-0.209,0.022,0.326,20.007
V9,283745.000,-0.002,1.096,-13.434,-0.644,-0.053,0.596,15.595


##### Data Visualization